# CIRRUS++ Master Benchmark: Synthetic Domains and Real Eye-Tracking Data

This notebook is the executable empirical basis for the CIRRUS++ recommender. It separates three concepts that must not be conflated:

| Concept | Meaning in CIRRUS++ | Current content |
|---|---|---|
| **Domain** | A domain-specific analysis objective and loss weighting | eye tracking, industrial, energy, physiological/wearable, finance, network |
| **Synthetic signal family** | A generated clean time series with known events and exact ground truth | one generator for each of the six domains |
| **Real dataset** | Concrete recorded files used to validate realistic signal profiles and preprocessing behavior | Autism, Desktop Activities, ETDD70/Dyslexia, EyeT4Empathy |

**Important:** the four real datasets currently all belong to the **eye-tracking domain**. The other five domains are currently evaluated with synthetic signals only. Therefore, the present evidence supports cross-domain *method development* and real-data validation within eye tracking; it does not yet prove real-world generalization to all other domains.

## Experimental logic

1. **Synthetic benchmark:** generate clean signals from six domain-specific families, inject known corruptions, and compare candidate methods against exact ground truth.
2. **Real eye-tracking benchmark:** load heterogeneous real recordings, map them to a canonical schema, extract fully observed windows without imputation, inject controlled corruptions, and compare recovery against the original observed window.
3. **Domain-aware evaluation:** raw measurements are identical across domains, but reconstruction, dynamics, frequency, event preservation, detection, and retention receive domain-specific weights.
4. **Rule induction:** combine synthetic and real winner tables and learn a shallow, auditable decision tree from profiling metrics such as missingness structure, autocorrelation, skewness, and excess kurtosis.
5. **External holdout:** keep one real dataset, e.g. ETDD70/Dyslexia, out of training and test whether rules transfer to it.

The real windows are **pseudo-clean references**, not claims that the complete recording is error-free. Only contiguous intervals with fully observed requested channels are used, and all evaluation corruptions are added afterwards with exact masks.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from cirruspp_benchmark.benchmark import load_config, run_stage_benchmark, summarize_winners
from cirruspp_benchmark.progress import NotebookProgress
from cirruspp_benchmark.real_data import load_real_dataset_specs
from cirruspp_benchmark.real_benchmark import (
    load_real_reference_windows,
    estimate_real_method_runs,
    run_real_stage_benchmark,
)
from cirruspp_benchmark.rule_induction import (
    fit_interpretable_rules,
    grouped_cv_accuracy,
    bootstrap_method_support,
    export_rule_bundle,
)

DATASET_CONFIG = Path(os.environ.get(
    "CIRRUS_DATASET_CONFIG",
    ROOT / "configs" / "real_datasets.local.yaml",
))
REAL_EXPERIMENT_CONFIG = ROOT / "configs" / "real_experiments_quick.yaml"
SYNTHETIC_EXPERIMENT_CONFIG = ROOT / "configs" / "experiments_quick.yaml"
DOMAIN_CONFIG = ROOT / "configs" / "domains.yaml"

RUN_REAL_BENCHMARK = True
RUN_SYNTHETIC_BENCHMARK = False  # set True to regenerate the packaged synthetic quick results

print("Project root:", ROOT)
print("Real dataset config:", DATASET_CONFIG)
print("Expected local data root: <set authorized local data directory>")

## 1. Load and canonicalize the four real datasets

The dataset-specific adapters handle the actual formats:

- **Autism:** binocular `Point of Regard` coordinates and pupil diameters; millisecond timestamps; blink/separator rows treated as missing.
- **Desktop Activities:** `x`, `y`, and millisecond timestamps.
- **ETDD70/Dyslexia:** averaged left/right gaze, pupil channels, and microsecond timestamps.
- **EyeT4Empathy:** Tobii gaze points, pupil channels with decimal commas, validity fields, and microsecond timestamps.

All files are mapped to `timestamp`, `gaze_x`, `gaze_y`, and optional pupil channels. Sampling rates are inferred per file from timestamps rather than blindly fixed.

In [ ]:
real_cfg = load_config(REAL_EXPERIMENT_CONFIG)
real_specs = load_real_dataset_specs(DATASET_CONFIG)
real_windows, real_inventory, real_failures = load_real_reference_windows(
    real_specs,
    feature_columns=tuple(real_cfg["feature_columns"]),
    window_seconds=float(real_cfg["window_seconds"]),
    max_windows_per_file=int(real_cfg["max_windows_per_file"]),
    min_window_samples=int(real_cfg["min_window_samples"]),
)

print(f"Loaded files: {len(real_inventory)}")
print(f"Extracted pseudo-clean reference windows: {len(real_windows)}")
real_inventory

In [ ]:
if not real_failures.empty:
    display(real_failures)
else:
    print("No loader or reference-window failures.")

window_summary = pd.DataFrame([
    {
        "dataset": w.metadata["dataset"],
        "file": w.metadata["source_file_name"],
        "window": w.metadata["reference_window"],
        "samples": len(w.clean),
        "seconds": w.metadata["reference_window_seconds"],
        "sampling_rate_hz": w.metadata["sampling_rate_hz"],
        "features": ", ".join(w.metadata["reference_features"]),
    }
    for w in real_windows
])
window_summary.groupby("dataset").agg(
    files=("file", "nunique"),
    windows=("window", "count"),
    median_samples=("samples", "median"),
    median_fs=("sampling_rate_hz", "median"),
)

## 2. Visual check of one reference window per real dataset

These are not preprocessed reconstructions. They are contiguous observed windows selected only because both gaze channels are present. They become the reference before controlled degradation.

In [ ]:
for dataset in sorted({w.metadata["dataset"] for w in real_windows}):
    w = next(x for x in real_windows if x.metadata["dataset"] == dataset)
    ax = w.clean.plot(x="timestamp", y=["gaze_x", "gaze_y"], figsize=(10, 3), title=f"{dataset}: observed reference window")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Gaze coordinate")
    plt.show()

## 3. Real eye-tracking benchmark with live percentage

For the quick run, each of the 20 supplied files contributes one six-second reference window. The benchmark injects MCAR missingness, block missingness, spikes, and high-frequency noise, then tests imputation, outlier, and smoothing methods. The progress display reports exact percentage, completed runs, elapsed time, ETA, dataset, corruption, stage, and method.

In [ ]:
REAL_OUTPUT = ROOT / "results" / "notebook_real_quick"
REAL_OUTPUT.mkdir(parents=True, exist_ok=True)

if RUN_REAL_BENCHMARK:
    total = estimate_real_method_runs(real_cfg, len(real_windows))
    print("Planned real-data method evaluations:", total)
    real_outputs = run_real_stage_benchmark(
        real_cfg,
        real_windows,
        DOMAIN_CONFIG,
        REAL_OUTPUT,
        progress_callback=NotebookProgress(display_id="cirruspp-real-progress"),
        checkpoint_every=100,
    )
    for stage, frame in real_outputs.items():
        winners = summarize_winners(frame)
        winners.to_csv(REAL_OUTPUT / f"real_{stage}_winners.csv", index=False)
else:
    real_outputs = {
        stage: pd.read_csv(REAL_OUTPUT / f"real_{stage}_benchmark.csv")
        for stage in ["imputation", "outlier", "smoothing"]
    }

print({stage: len(frame) for stage, frame in real_outputs.items()})

### Quick-run diagnostics — not final paper claims

A single seed, one window per file, and one severity per corruption are sufficient to validate the machinery, not to freeze the final rules. The following table is deliberately descriptive.

In [ ]:
real_winners = {}
for stage in ["imputation", "outlier", "smoothing"]:
    path = REAL_OUTPUT / f"real_{stage}_winners.csv"
    w = pd.read_csv(path) if path.exists() else summarize_winners(real_outputs[stage])
    real_winners[stage] = w
    print(f"\n{stage.upper()}")
    display(pd.crosstab(w["dataset"], w["winner"]))

## 4. Synthetic cross-domain benchmark

The six synthetic domains are generated in code; they are not folders under the real-data path. Each generator creates a clean signal family with domain-typical structure and genuine events:

- eye tracking: fixations, saccades, and pupil dynamics;
- industrial: cycles and operating-state changes;
- energy: daily/weekly seasonality and demand peaks;
- physiological/wearable: quasi-periodic channels and motion episodes;
- finance: heavy-tailed returns, volatility regimes, and genuine jumps;
- network: load cycles, bursts, and incidents.

This branch tests whether the rule-learning mechanism behaves sensibly across different signal structures. Real non-eye-tracking corpora can be added later through the same adapter interface.

In [ ]:
SYNTHETIC_OUTPUT = ROOT / "results" / "notebook_synthetic_quick"
SYNTHETIC_OUTPUT.mkdir(parents=True, exist_ok=True)
synthetic_cfg = load_config(SYNTHETIC_EXPERIMENT_CONFIG)

if RUN_SYNTHETIC_BENCHMARK:
    synthetic_outputs = run_stage_benchmark(
        synthetic_cfg,
        DOMAIN_CONFIG,
        SYNTHETIC_OUTPUT,
        progress_callback=NotebookProgress(display_id="cirruspp-synthetic-progress"),
        checkpoint_every=100,
    )
    for stage, frame in synthetic_outputs.items():
        summarize_winners(frame).to_csv(SYNTHETIC_OUTPUT / f"{stage}_winners.csv", index=False)
else:
    packaged = ROOT / "results" / "quick"
    synthetic_winner_dir = packaged if (packaged / "imputation_winners.csv").exists() else SYNTHETIC_OUTPUT
    print("Using existing synthetic winners from:", synthetic_winner_dir)

## 5. Combine evidence and induce auditable rules

The rule model uses profiling metrics, corruption type, declared domain, and source type. It **does not use the dataset name as a predictor**, so it cannot simply memorize that ETDD70 prefers a method. Here ETDD70/Dyslexia is held out as an external real dataset.

In [ ]:
COMBINED_OUTPUT = ROOT / "results" / "notebook_combined"
ARTIFACT_OUTPUT = ROOT / "artifacts" / "notebook_combined"
COMBINED_OUTPUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_OUTPUT.mkdir(parents=True, exist_ok=True)

combined_tables = {}
for stage in ["imputation", "outlier", "smoothing"]:
    # locate synthetic winner table (packaged naming is stage_winners.csv)
    candidates = [
        ROOT / "results" / "quick" / f"{stage}_winners.csv",
        SYNTHETIC_OUTPUT / f"{stage}_winners.csv",
    ]
    synthetic_path = next((p for p in candidates if p.exists()), None)
    synthetic = pd.read_csv(synthetic_path) if synthetic_path else pd.DataFrame()
    if not synthetic.empty:
        synthetic["source_type"] = "synthetic"
        if "dataset" not in synthetic:
            synthetic["dataset"] = "synthetic_" + synthetic["domain"].astype(str)
    real = real_winners[stage].copy()
    real["source_type"] = "real"
    combined = pd.concat([synthetic, real], ignore_index=True, sort=False)
    combined_tables[stage] = combined
    combined.to_csv(COMBINED_OUTPUT / f"combined_{stage}_winners.csv", index=False)

    result = fit_interpretable_rules(
        combined,
        holdout_column="dataset",
        holdout_value="etdd70_dyslexia",
    )
    export_rule_bundle(
        result,
        ARTIFACT_OUTPUT,
        stage,
        metadata={"real_data_config": str(DATASET_CONFIG)},
    )
    cv = grouped_cv_accuracy(combined)
    cv.to_csv(ARTIFACT_OUTPUT / f"{stage}_grouped_cv.csv", index=False)
    support = bootstrap_method_support(combined, n_boot=100)
    support.to_csv(ARTIFACT_OUTPUT / f"{stage}_bootstrap_support.csv", index=False)
    print(stage, "rows=", len(combined), "holdout accuracy=", result.get("holdout_accuracy"))
    print(result["tree_text"][:1800])

## 6. What the current evidence does and does not establish

The final paper should distinguish the following claims:

- **Supported after the full run:** CIRRUS++ can derive transparent preprocessing recommendations from controlled corruptions; the rules can be validated across heterogeneous real eye-tracking datasets and tested for sensitivity to domain weights.
- **Supported as a design demonstration:** the same profiling, corruption, loss, and rule-induction machinery can operate on synthetic signal families from several time-series domains.
- **Not yet supported:** universal effectiveness on real industrial, financial, energy, physiological, and network datasets. At least one or two real non-eye-tracking corpora should be added before making a strong cross-domain empirical claim.

The next publication run should use all permitted files, multiple windows per file, five seeds, multiple severities, one untouched real dataset holdout, and a weight-sensitivity analysis. The quick run generated here is a validated execution path, not the final result table.